In [1]:
import numpy as np
import pandas as pd
import seaborn as sns

In [2]:
movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")

In [3]:
movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800


In [4]:
credits.head(1)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [5]:
movies = movies.merge(credits)

In [6]:
q1 = movies["popularity"].quantile(0.25)
q3 = movies["popularity"].quantile(0.75)
IQR = q3-q1
upper = q3 + IQR*1.5
lower = q1 - IQR*1.5
movies.loc[(upper<movies["popularity"]),"popularity"] = upper
movies.loc[(lower > movies["popularity"]),"popularity"] = lower


In [7]:
movies["popularity"] = pd.cut(movies["popularity"], 
                         bins=[0, 10, 40, 70], 
                         labels=['Low', 'Medium', 'High'])

In [8]:
movies["rating"] = round((movies["vote_count"]/(movies["vote_count"]+movies["vote_count"].mean())
                    *movies["vote_average"])+(movies["vote_count"].mean()/(movies["vote_count"]+
                     movies["vote_count"].mean())
                    *movies["vote_average"].mean()),2)

In [9]:
movies.head(2)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew,rating
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",High,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",7.14
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",High,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,285,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",6.79


In [10]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype   
---  ------                --------------  -----   
 0   budget                4809 non-null   int64   
 1   genres                4809 non-null   object  
 2   homepage              1713 non-null   object  
 3   id                    4809 non-null   int64   
 4   keywords              4809 non-null   object  
 5   original_language     4809 non-null   object  
 6   original_title        4809 non-null   object  
 7   overview              4806 non-null   object  
 8   popularity            4808 non-null   category
 9   production_companies  4809 non-null   object  
 10  production_countries  4809 non-null   object  
 11  release_date          4808 non-null   object  
 12  revenue               4809 non-null   int64   
 13  runtime               4807 non-null   float64 
 14  spoken_languages      4809 non-null   object  
 15  stat

In [11]:
#generes
#id
#keywords
#overview
#popularity
#title
#cast
#crew
#rating

In [12]:
movies = movies[["genres","id","keywords","overview","popularity","title","cast","crew","rating"]]

In [13]:
movies.head(1)

,genres,id,keywords,overview,popularity,title,cast,crew,rating
0,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","In the 22nd century, a paraplegic Marine is di...",High,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",7.14


In [14]:
movies.isnull().sum()

genres        0
id            0
keywords      0
overview      3
popularity    1
title         0
cast          0
crew          0
rating        0
dtype: int64

In [15]:
movies = movies.dropna()

In [16]:
movies.duplicated().sum()

np.int64(0)

In [17]:
import ast

In [18]:
def convert(x):
    l = []
    x = ast.literal_eval(x)
    for i in x:
        l.append(i["name"])
    return l    

In [19]:
movies["genres"] = movies["genres"].apply(convert)
movies["keywords"] = movies["keywords"].apply(convert)

In [20]:
def convertcast(x):
    a = 0
    l = []
    x = ast.literal_eval(x)
    for i in x:
        if a == 3:
            break
        a+=1
        l.append(i["name"])
    return l

In [21]:
movies["cast"] = movies["cast"].apply(convertcast)

In [22]:
def convertcrew(x):
    l = []
    x = ast.literal_eval(x)
    for i in x:
        if i["job"]=="Director":
            l.append(i["name"])
            break
    return l

In [23]:
movies["crew"] = movies["crew"].apply(convertcrew)

In [24]:
movies["overview"] = movies["overview"].apply(lambda x: x.split())

In [25]:
movies["crew"] = movies["crew"].apply(lambda x:[i.replace(" ","") for i in x])
movies["genres"] = movies["genres"].apply(lambda x:[i.replace(" ","") for i in x])
movies["keywords"] = movies["keywords"].apply(lambda x:[i.replace(" ","") for i in x])
movies["cast"] = movies["cast"].apply(lambda x:[i.replace(" ","") for i in x])
movies.head(1)

,genres,id,keywords,overview,popularity,title,cast,crew,rating
0,"[Action, Adventure, Fantasy, ScienceFiction]",19995,"[cultureclash, future, spacewar, spacecolony, ...","[In, the, 22nd, century,, a, paraplegic, Marin...",High,Avatar,"[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron],7.14


In [26]:
movies["keywords"] = movies.apply(
    lambda row: row["keywords"] + [row["popularity"],row["rating"]],
    axis=1
)

In [27]:
movies["tags"] = movies["genres"] + movies["keywords"] + movies["overview"] + movies["cast"] + movies["crew"]

In [28]:
new_df = movies[["id","title","tags"]]

In [29]:
new_df["tags"] = new_df["tags"].apply(
    lambda x: " ".join(map(str, x))
)

C:\Users\Asus\AppData\Local\Temp\ipykernel_18692\300749695.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df["tags"] = new_df["tags"].apply(


In [30]:
new_df.head()

,id,title,tags
0,19995,Avatar,Action Adventure Fantasy ScienceFiction cultur...
1,285,Pirates of the Caribbean: At World's End,Adventure Fantasy Action ocean drugabuse exoti...
2,206647,Spectre,Action Adventure Crime spy basedonnovel secret...
3,49026,The Dark Knight Rises,Action Crime Drama Thriller dccomics crimefigh...
4,49529,John Carter,Action Adventure ScienceFiction basedonnovel m...


In [31]:
from sklearn.feature_extraction.text import CountVectorizer

In [32]:
cv = CountVectorizer(max_features=5000,stop_words="english")

In [33]:
vectors = cv.fit_transform(new_df["tags"]).toarray()

In [34]:
vectors

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(4805, 5000))

In [35]:
import nltk

In [36]:
from nltk.stem.porter import PorterStemmer

In [37]:
ps = PorterStemmer()

In [38]:
def stem(text):
    l = []
    for i in text.split():
        l.append(ps.stem(i))
    return " ".join(l)

In [39]:
new_df["tags"]=new_df["tags"].apply(stem)

C:\Users\Asus\AppData\Local\Temp\ipykernel_18692\1027202188.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df["tags"]=new_df["tags"].apply(stem)


In [40]:
new_df.head()

,id,title,tags
0,19995,Avatar,action adventur fantasi sciencefict culturecla...
1,285,Pirates of the Caribbean: At World's End,adventur fantasi action ocean drugabus exotici...
2,206647,Spectre,action adventur crime spi basedonnovel secreta...
3,49026,The Dark Knight Rises,action crime drama thriller dccomic crimefight...
4,49529,John Carter,action adventur sciencefict basedonnovel mar m...


In [41]:
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vectors)

In [42]:
def recommend(movie):
    movie_index = new_df[new_df["title"]==movie].index[0]
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:6]
    for i in movies_list:
        print(new_df.iloc[i[0]].title)

In [43]:
recommend("Avatar")

Small Soldiers
Ender's Game
Independence Day
Battle: Los Angeles
Aliens


In [46]:
import pickle
pickle.dump(new_df.to_dict(),open("movies_dict.pkl","wb"))

In [47]:
pickle.dump(similarity,open("similarity.pkl","wb"))